# RAG

In [1]:
# Minimal RAG with OpenAI Agents SDK + cosine similarity (in-memory)
# Deps: pip install openai-agents openai numpy
import os
import uuid
import math
import asyncio
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
from openai import OpenAI
from agents import Agent, Runner, function_tool

# ---------- Config ----------
EMBED_MODEL = "text-embedding-3-small"  # fast & cheap; 1536-d
client = OpenAI()  # uses OPENAI_API_KEY env

In [2]:
# ---------- Tiny vector store ----------
class VectorStore:
    """
    Very small in-memory store: holds chunks, metadata, normalized embeddings.
    """
    def __init__(self):
        self.chunk_ids: List[str] = []
        self.texts: List[str] = []
        self.titles: List[str] = []
        self.sources: List[str] = []
        self.embs: Optional[np.ndarray] = None  # shape: (N, D)

    def clear(self):
        self.chunk_ids, self.texts, self.titles, self.sources = [], [], [], []
        self.embs = None

    @property
    def size(self) -> int:
        return len(self.texts)

STORE = VectorStore()

# ---------- Helpers ----------
def _chunk_text(text: str, max_chars: int = 1200, overlap: int = 100) -> List[str]:
    """
    Simple, layout-preserving chunker: split on paragraphs, then pack up to max_chars.
    Overlap to keep context continuity.
    """
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, buf = [], ""
    for p in paras:
        if len(buf) + len(p) + 1 <= max_chars:
            buf = (buf + "\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            # start next buffer with overlap tail of previous
            tail = buf[-overlap:] if overlap and buf else ""
            buf = (tail + "\n" + p).strip()
    if buf:
        chunks.append(buf)
    return chunks or [text[:max_chars]]

def _embed(texts: List[str]) -> np.ndarray:
    """
    OpenAI embeddings → np.array, L2-normalized for cosine similarity.
    """
    if not texts:
        return np.zeros((0, 1536), dtype=np.float32)
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    arr = np.array([d.embedding for d in resp.data], dtype=np.float32)
    # L2 normalize for cosine similarity = dot product
    norms = np.linalg.norm(arr, axis=1, keepdims=True) + 1e-12
    return arr / norms

def _cosine_topk(query_vec: np.ndarray, mat: np.ndarray, k: int) -> List[Tuple[int, float]]:
    """
    query_vec: (D,), mat: (N, D) normalized → cosine = dot.
    Returns list of (index, score) sorted desc.
    """
    if mat is None or mat.shape[0] == 0:
        return []
    sims = mat @ query_vec  # (N,)
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

In [3]:
# ---------- Tools ----------
@function_tool
def rag_clear() -> str:
    """Clear the in-memory store."""
    STORE.clear()
    return "Cleared."

@function_tool
def rag_ingest(title: str, text: str, source: str = "") -> Dict[str, Any]:
    """
    Ingest a document: chunk → embed → store.
    Returns how many chunks were added and their IDs.
    """
    chunks = _chunk_text(text, max_chars=1200, overlap=120)
    embs = _embed(chunks)

    ids = []
    for chunk, emb in zip(chunks, embs):
        cid = str(uuid.uuid4())
        STORE.chunk_ids.append(cid)
        STORE.texts.append(chunk)
        STORE.titles.append(title)
        STORE.sources.append(source or title)
        ids.append(cid)

    # Append embeddings into matrix
    STORE.embs = embs if STORE.embs is None else np.vstack([STORE.embs, embs])
    return {"added_chunks": len(ids), "chunk_ids": ids}

@function_tool
def rag_search(query: str, k: int = 4) -> List[Dict[str, Any]]:
    """
    Retrieve top-k relevant chunks using cosine similarity.
    Returns [{chunk_id, title, source, text, score}]
    """
    qv = _embed([query])[0]  # (D,)
    top = _cosine_topk(qv, STORE.embs, k)
    results: List[Dict[str, Any]] = []
    for i, score in top:
        results.append({
            "chunk_id": STORE.chunk_ids[i],
            "title": STORE.titles[i],
            "source": STORE.sources[i],
            "text": STORE.texts[i],
            "score": round(score, 4),
        })
    return results

In [4]:
# ---------- Agent ----------
rag_agent = Agent(
    name="Mini RAG",
    instructions=(
        "You are a grounded assistant over a small local knowledge base.\n"
        "ALWAYS call rag_search with the user's question before answering.\n"
        "Use ONLY the returned chunks as evidence. If evidence is weak or empty, say so.\n"
        "Answer concisely with bullet points and include a 'References' section with numbered sources.\n"
        "Citations format: [1], [2] inline, mapping to the order in 'References'."
    ),
    tools=[rag_search, rag_ingest, rag_clear],
)

In [5]:
# ---------- Demo (optional) ----------
# 1) reset
print(await Runner.run(rag_agent, "Clear the store please by calling rag_clear."))
await Runner.run(rag_agent, "rag_clear")  # or call tool directly if you prefer

# 2) ingest two tiny docs
await Runner.run(
    rag_agent,
    "Ingest this doc: title='Python Loops', source='docs', text='In Python, for loops iterate over iterables. Use range(n) for counts. break/continue control flow.'"
)
await Runner.run(
    rag_agent,
    "Ingest this doc: title='While Loop Basics', source='docs', text='A while loop repeats while a condition is True. Beware infinite loops; update the condition.'"
)

# 3) ask a question; the agent will call rag_search then answer with citations
res = await Runner.run(rag_agent, "How do I loop in Python and avoid infinite loops?")
print(res.final_output)

# If running as a script:
# asyncio.run(demo())

RunResult:
- Last agent: Agent(name="Mini RAG", ...)
- Final output (str):
    The store has been cleared.
- 3 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
- **While Loop**: In Python, a `while` loop continues to execute as long as the specified condition is `True`. To avoid infinite loops, ensure the condition can become `False` by updating variables involved in the condition within the loop [1].
  
- **For Loop**: Python's `for` loop iterates over elements in a sequence, such as lists or ranges. Using `for` loops can help avoid infinite loops as they automatically end when the sequence is exhausted. Using constructs like `range(n)` allows looping a specific number of times [2].

- **Control Flow**: Utilize `break` to exit a loop prematurely based on a condition, and `continue` to skip the rest of the loop body for the current iteration, proceeding to the next iteration immediately [2].

**References**: